TAHAP 1

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 38.3 MB/s eta 0:00:00


In [4]:
!pip install pdfminer.six

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 48.6 MB/s eta 0:00:00


In [7]:
import os
from pdfminer.high_level import extract_text
import re

# Buat folder untuk hasil
os.makedirs("/content/drive/MyDrive/TGS PK/data/raw", exist_ok=True)
os.makedirs("/content/drive/MyDrive/TGS PK/logs", exist_ok=True)

# Fungsi membersihkan teks
def clean_text(text):
    text = re.sub(r"(\nPage\s+\d+|\nHalaman\s+\d+|\n\d+\s+/\s+\d+)", "", text)  # Hapus header/footer
    text = text.lower()  # Ubah ke lowercase
    text = re.sub(r"\s+", " ", text)  # Normalisasi spasi
    text = re.sub(r"[^\w\s]", "", text)  # Hapus tanda baca
    return text.strip()

# Fungsi ekstrak dan bersihkan PDF
def convert_and_clean_pdf(pdf_path, output_txt_path):
    try:
        raw_text = extract_text(pdf_path)
        if not raw_text or len(raw_text.strip()) < 100:
            return False, "Teks terlalu pendek atau kosong"

        cleaned_text = clean_text(raw_text)

        # Validasi 80%
        if len(cleaned_text) < 0.8 * len(raw_text):
            return False, "Validasi gagal (kurang dari 80%)"

        with open(output_txt_path, "w", encoding="utf-8") as f:
            f.write(cleaned_text)

        return True, f"Sukses ({len(cleaned_text)} karakter)"
    except Exception as e:
        return False, f"ERROR: {str(e)}"

In [8]:
source_folder = "/content/drive/MyDrive/TGS PK/Kumpulan Putusan"
output_folder = "/content/drive/MyDrive/TGS PK/data/raw"
log_path = "/content/drive/MyDrive/TGS PK/logs/cleaning.log"

log_lines = []

for i, filename in enumerate(sorted(os.listdir(source_folder))):
    if filename.lower().endswith(".pdf"):
        case_id = f"case_{i+1:03}"
        pdf_path = os.path.join(source_folder, filename)
        txt_path = os.path.join(output_folder, f"{case_id}.txt")

        status, message = convert_and_clean_pdf(pdf_path, txt_path)
        log_lines.append(f"{case_id} - {filename} - {message}")

# Simpan log
with open(log_path, "w", encoding="utf-8") as f:
    f.write("\n".join(log_lines))

print("✅ Proses selesai. Cek folder:")
print("- /TGS PK/data/raw/")
print("- /TGS PK/logs/cleaning.log")

✅ Proses selesai. Cek folder:
- /TGS PK/data/raw/
- /TGS PK/logs/cleaning.log


TAHAP 2

In [9]:
import re

def extract_metadata(text):
    metadata = {}
    metadata['no_perkara'] = re.search(r'Nomor\s*[:\-]?\s*(.*)', text)
    metadata['tanggal'] = re.search(r'Tanggal\s*[:\-]?\s*(\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4})', text)
    metadata['jenis_perkara'] = re.search(r'Jenis Perkara\s*[:\-]?\s*(.*)', text)
    metadata['pasal'] = re.findall(r'Pasal\s+\d+\s+\w+', text)
    metadata['pihak'] = re.findall(r'(Terdakwa|Jaksa|Penasehat Hukum).*', text)

    # Bersihkan hasil
    for k, v in metadata.items():
        if isinstance(v, re.Match):
            metadata[k] = v.group(1).strip()
        elif isinstance(v, list):
            metadata[k] = ', '.join([i.strip() for i in v])
        else:
            metadata[k] = ''
    return metadata

In [10]:
def extract_fakta_dan_putusan(text):
    fakta = re.search(r'(Fakta-fakta|Pertimbangan).*?(?=Pertimbangan Hukum|Amar Putusan)', text, re.DOTALL)
    putusan = re.search(r'(Amar Putusan|Putusan).*', text, re.DOTALL)

    return {
        'ringkasan_fakta': fakta.group(0).strip() if fakta else '',
        'argumen_hukum': putusan.group(0).strip() if putusan else ''
    }

In [11]:
def feature_engineering(text):
    return {
        'length_words': len(text.split()),
        'length_chars': len(text)
    }

In [12]:
import os
import pandas as pd
import re

raw_dir = "/content/drive/MyDrive/TGS PK/data/raw"
processed_path = "/content/drive/MyDrive/TGS PK/data/processed"
os.makedirs(processed_path, exist_ok=True)

data = []

for filename in os.listdir(raw_dir):
    if filename.endswith(".txt"):
        with open(os.path.join(raw_dir, filename), "r", encoding="utf-8") as f:
            text = f.read()

            # --- Metadata ---
            no_perkara = re.search(r"\d+/\w+/?\w*/\d{4}", text)
            tanggal = re.search(r"(\d{1,2}\s+\w+\s+\d{4})", text)
            pasal = "; ".join(re.findall(r"Pasal\s+\d+[^.\n]*", text))
            pihak = re.findall(r"([A-Z][a-z]+)\s+(?:melawan|vs\.?)\s+([A-Z][a-z]+)", text)

            # --- Ringkasan Fakta Sederhana ---
            ringkasan = text[:500].strip()

            data.append({
                "case_id": filename.replace(".txt", ""),
                "no_perkara": no_perkara.group(0) if no_perkara else "-",
                "tanggal": tanggal.group(1) if tanggal else "-",
                "ringkasan_fakta": ringkasan,
                "pasal": pasal if pasal else "-",
                "pihak": " vs. ".join(pihak[0]) if pihak else "-",
                "text_full": text[:300] + "..."  # preview potongan awal
            })

# Simpan sebagai CSV
df = pd.DataFrame(data)
csv_path = os.path.join(processed_path, "/content/drive/MyDrive/TGS PK/data/processed/cases.csv")
df.to_csv(csv_path, index=False, encoding="utf-8")

print(f"✅ File berhasil disimpan di: {csv_path}")

✅ File berhasil disimpan di: /content/drive/MyDrive/TGS PK/data/processed/cases.csv


In [23]:
import json
import pandas as pd
import os
import re
import time
import hashlib
import numpy as np

# ======================================================
# FUNGSI EKSTRAKSI (CONTOH - HARUS DIKEMBANGKAN SESUAI KEBUTUHAN)
# ======================================================
def extract_metadata(full_text):
    """Ekstrak metadata dari teks putusan (contoh sederhana)"""
    meta = {
        'no_perkara': "Tidak Diketahui",
        'tanggal': "Tidak Diketahui",
        'jenis_perkara': "Tidak Diketahui",
        'pasal': [],
        'pihak': []
    }

    try:
        # Contoh pola untuk nomor perkara (format: XXX/YYY/TAHUN)
        no_match = re.search(r'Nomor\s*:\s*([\d/]+/\w+/\d{4})', full_text)
        if no_match:
            meta['no_perkara'] = no_match.group(1).strip()

        # Contoh pola untuk tanggal (format DD MMMM YYYY)
        tgl_match = re.search(r'Tanggal\s*:\s*(\d{1,2}\s+[A-Za-z]+\s+\d{4})', full_text)
        if tgl_match:
            meta['tanggal'] = tgl_match.group(1).strip()

        # Contoh pola untuk jenis perkara
        jenis_match = re.search(r'Jenis\s+Perkara\s*:\s*([\w\s]+)', full_text, re.IGNORECASE)
        if jenis_match:
            meta['jenis_perkara'] = jenis_match.group(1).strip()

        # Contoh pola untuk pasal (sederhana)
        pasal_matches = re.findall(r'(Pasal\s+\d+\s+[A-Za-z]+)', full_text)
        if pasal_matches:
            meta['pasal'] = list(set(pasal_matches))  # Hapus duplikat

        # Contoh pola untuk pihak (sederhana)
        pihak_matches = re.findall(r'(Penggugat|Tergugat|Pemohon|Termohon)\s*:\s*([^\n]+)', full_text)
        if pihak_matches:
            meta['pihak'] = [f"{p[0]}: {p[1].strip()}" for p in pihak_matches]

    except Exception as e:
        print(f"⚠️ Error saat ekstraksi metadata: {str(e)}")

    return meta

def extract_fakta_dan_putusan(full_text):
    """Ekstrak bagian fakta dan putusan (contoh sederhana)"""
    try:
        # Cari bagian FAKTA
        fakta_match = re.search(r'FAKTA HUKUM(.*?)HUKUM', full_text, re.DOTALL | re.IGNORECASE)
        ringkasan_fakta = fakta_match.group(1).strip()[:1000] + "..." if fakta_match else full_text[:500] + "..."

        # Cari bagian PUTUSAN
        putusan_match = re.search(r'(PERTIMBANGAN HUKUM.*?)(?:PUTUSAN|$)', full_text, re.DOTALL | re.IGNORECASE)
        argumen_hukum = putusan_match.group(1).strip()[:1000] + "..." if putusan_match else full_text[-500:] + "..."

    except Exception as e:
        print(f"⚠️ Error saat ekstraksi konten: {str(e)}")
        ringkasan_fakta = full_text[:500] + "..."
        argumen_hukum = full_text[-500:] + "..."

    return {
        'ringkasan_fakta': ringkasan_fakta,
        'argumen_hukum': argumen_hukum
    }

def feature_engineering(full_text):
    """Hitung fitur dasar dari teks"""
    words = full_text.split()
    chars = full_text

    return {
        'length_words': len(words),
        'length_chars': len(chars),
        'avg_word_length': sum(len(word) for word in words)/len(words) if words else 0
    }

def process_case(case_id, full_text, file_name=""):
    """Proses satu kasus lengkap"""
    try:
        print(f"\n📄 Memproses kasus {case_id} ({file_name})...")
        start_time = time.time()

        meta = extract_metadata(full_text)
        content = extract_fakta_dan_putusan(full_text)
        features = feature_engineering(full_text)

        processing_time = time.time() - start_time

        # Konversi list ke string untuk kolom 'pasal' dan 'pihak'
        pasal_str = "; ".join(meta['pasal']) if isinstance(meta['pasal'], list) else meta['pasal']
        pihak_str = "; ".join(meta['pihak']) if isinstance(meta['pihak'], list) else meta['pihak']

        result = {
            "case_id": case_id,
            "file_name": file_name,
            "no_perkara": meta['no_perkara'],
            "tanggal": meta['tanggal'],
            "jenis_perkara": meta['jenis_perkara'],
            "pasal": pasal_str,  # Sekarang string, bukan list
            "pihak": pihak_str,   # Sekarang string, bukan list
            "ringkasan_fakta": content['ringkasan_fakta'],
            "argumen_hukum": content['argumen_hukum'],
            "length_words": features['length_words'],
            "length_chars": features['length_chars'],
            "avg_word_length": features['avg_word_length'],
            "processing_time_sec": round(processing_time, 2),
            "text_md5": hashlib.md5(full_text.encode('utf-8')).hexdigest(),
            "text_full": full_text[:300] + "..."  # Simpan hanya preview teks
        }

        print(f"✅ Selesai dalam {processing_time:.2f} detik")
        return result

    except Exception as e:
        print(f"❌ Gagal memproses kasus {case_id}: {str(e)}")
        return None

# ======================================================
# KONFIGURASI UTAMA
# ======================================================
INPUT_DIR = "/content/drive/MyDrive/TGS PK/data/raw"
OUTPUT_DIR = "/content/drive/MyDrive/TGS PK/data/processed"
LOG_FILE = os.path.join(OUTPUT_DIR, "processing_log.txt")

# Buat direktori jika belum ada
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Setup log file
with open(LOG_FILE, "w", encoding="utf-8") as log:
    log.write("LOG PROSES EKSTRAKSI PUTUSAN PENGADILAN\n")
    log.write(f"Dimulai pada: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    log.write("=" * 50 + "\n")

# ======================================================
# BACA FILE INPUT
# ======================================================
print(f"\n🔍 Mencari file di {INPUT_DIR}...")
file_list = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.txt', '.pdf'))]

# Prioritaskan file txt
txt_files = [f for f in file_list if f.lower().endswith('.txt')]
pdf_files = [f for f in file_list if f.lower().endswith('.pdf')]
sorted_files = txt_files + pdf_files

print(f"📂 Ditemukan {len(sorted_files)} file:")
print(f" - Text: {len(txt_files)} file")
print(f" - PDF: {len(pdf_files)} file")

# Simpan ke log
with open(LOG_FILE, "a", encoding="utf-8") as log:
    log.write(f"Total file ditemukan: {len(sorted_files)}\n")
    log.write(f"Detail file:\n")
    for i, f in enumerate(sorted_files[:5]):
        log.write(f"  {i+1}. {f}\n")
    if len(sorted_files) > 5:
        log.write(f"  ... dan {len(sorted_files)-5} file lainnya\n")

# ======================================================
# PROSES FILE
# ======================================================
print(f"\n⏳ Memulai pemrosesan {len(sorted_files)} file...")
start_total_time = time.time()
data_all = []
failed_files = []

for i, file_name in enumerate(sorted_files):
    file_path = os.path.join(INPUT_DIR, file_name)
    case_id = f"PN{i+1:04d}"  # Format ID dengan leading zeros (PN0001)

    try:
        # Baca file
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()

        # Proses kasus
        case_data = process_case(case_id, text, file_name)

        if case_data:
            data_all.append(case_data)
        else:
            failed_files.append(file_name)
            with open(LOG_FILE, "a", encoding="utf-8") as log:
                log.write(f"❌ GAGAL: {file_name}\n")

    except UnicodeDecodeError:
        try:
            # Coba encoding alternatif
            with open(file_path, 'r', encoding='latin-1') as f:
                text = f.read()

            case_data = process_case(case_id, text, file_name)

            if case_data:
                data_all.append(case_data)
                with open(LOG_FILE, "a", encoding="utf-8") as log:
                    log.write(f"⚠️ PERINGATAN: {file_name} menggunakan encoding latin-1\n")
            else:
                failed_files.append(file_name)
                with open(LOG_FILE, "a", encoding="utf-8") as log:
                    log.write(f"❌ GAGAL: {file_name} (setelah coba latin-1)\n")

        except Exception as e:
            failed_files.append(file_name)
            print(f"❌ Gagal membaca {file_name}: {str(e)}")
            with open(LOG_FILE, "a", encoding="utf-8") as log:
                log.write(f"❌ GAGAL: {file_name} - {str(e)}\n")

    except Exception as e:
        failed_files.append(file_name)
        print(f"❌ Gagal memproses {file_name}: {str(e)}")
        with open(LOG_FILE, "a", encoding="utf-8") as log:
            log.write(f"❌ GAGAL: {file_name} - {str(e)}\n")

    # Progress report setiap 10 file
    if (i+1) % 10 == 0 or (i+1) == len(sorted_files):
        print(f"⏱️ Progress: {i+1}/{len(sorted_files)} file ({len(data_all)} sukses, {len(failed_files)} gagal)")

# Hitung waktu total
total_time = time.time() - start_total_time
print(f"\n✅ Semua file selesai diproses dalam {total_time:.2f} detik")

# ======================================================
# SIMPAN HASIL
# ======================================================
print("\n💾 Menyimpan hasil...")
# Simpan sebagai CSV
csv_path = os.path.join(OUTPUT_DIR, "cases.csv")
df = pd.DataFrame(data_all)

# Reorder kolom untuk memudahkan
column_order = [
    'case_id', 'file_name', 'no_perkara', 'tanggal', 'jenis_perkara',
    'length_words', 'length_chars', 'avg_word_length', 'processing_time_sec',
    'text_md5'
]
# Pastikan hanya kolom yang ada yang ditambahkan
available_columns = [col for col in column_order if col in df.columns]
other_columns = [col for col in df.columns if col not in available_columns]
df = df[available_columns + other_columns]

df.to_csv(csv_path, index=False, encoding='utf-8')
print(f"  - CSV disimpan: {csv_path} ({len(df)} baris)")

# Simpan sebagai JSON
json_path = os.path.join(OUTPUT_DIR, "cases.json")
with open(json_path, "w", encoding='utf-8') as f:
    json.dump(data_all, f, ensure_ascii=False, indent=2)
print(f"  - JSON disimpan: {json_path}")

# Simpan data gagal
if failed_files:
    failed_path = os.path.join(OUTPUT_DIR, "failed_files.txt")
    with open(failed_path, "w", encoding='utf-8') as f:
        f.write("\n".join(failed_files))
    print(f"  - Daftar file gagal disimpan: {failed_path}")

# ======================================================
# LAPORAN AKHIR
# ======================================================
print("\n📊 LAPORAN AKHIR:")
print(f"Total file diproses : {len(sorted_files)}")
print(f"File berhasil       : {len(data_all)}")
print(f"File gagal          : {len(failed_files)}")
print(f"Waktu total         : {total_time:.2f} detik")
print(f"Rata-rata per file  : {total_time/len(sorted_files):.2f} detik" if sorted_files else "N/A")

# Verifikasi checksum
if data_all:
    try:
        # Gunakan metode alternatif untuk checksum
        combined_text = ''.join([str(x) for x in df.values.flatten()])
        checksum = hashlib.md5(combined_text.encode('utf-8')).hexdigest()
        print(f"Checksum data      : {checksum}")
    except Exception as e:
        print(f"⚠️ Gagal menghitung checksum: {str(e)}")

# Tampilkan preview
print("\n🔍 Preview data:")
if data_all:
    # Pilih kolom yang pasti ada
    preview_cols = ['case_id', 'file_name', 'no_perkara', 'tanggal', 'length_words']
    available_cols = [col for col in preview_cols if col in df.columns]

    if available_cols:
        preview_df = df.head(3)[available_cols]
        print(preview_df)
    else:
        print("Kolom preview tidak tersedia")
else:
    print("Tidak ada data untuk ditampilkan")

# Update log file
with open(LOG_FILE, "a", encoding="utf-8") as log:
    log.write("\n" + "=" * 50 + "\n")
    log.write(f"Selesai pada: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    log.write(f"Total file diproses : {len(sorted_files)}\n")
    log.write(f"File berhasil       : {len(data_all)}\n")
    log.write(f"File gagal          : {len(failed_files)}\n")
    log.write(f"Waktu total         : {total_time:.2f} detik\n")
    if data_all:
        try:
            # Hitung checksum untuk log
            combined_text = ''.join([str(x) for x in df.values.flatten()])
            checksum_log = hashlib.md5(combined_text.encode('utf-8')).hexdigest()
            log.write(f"Checksum data       : {checksum_log}\n")
        except:
            log.write("Checksum data       : Gagal dihitung\n")
    log.write("=" * 50 + "\n")

print("\n✅ Semua proses selesai!")


🔍 Mencari file di /content/drive/MyDrive/TGS PK/data/raw...
📂 Ditemukan 30 file:
 - Text: 30 file
 - PDF: 0 file

⏳ Memulai pemrosesan 30 file...

📄 Memproses kasus PN0001 (case_001.txt)...
✅ Selesai dalam 0.02 detik

📄 Memproses kasus PN0002 (case_002.txt)...
✅ Selesai dalam 0.02 detik

📄 Memproses kasus PN0003 (case_003.txt)...
✅ Selesai dalam 0.01 detik

📄 Memproses kasus PN0004 (case_004.txt)...
✅ Selesai dalam 0.01 detik

📄 Memproses kasus PN0005 (case_005.txt)...
✅ Selesai dalam 0.01 detik

📄 Memproses kasus PN0006 (case_006.txt)...
✅ Selesai dalam 0.01 detik

📄 Memproses kasus PN0007 (case_007.txt)...
✅ Selesai dalam 0.01 detik

📄 Memproses kasus PN0008 (case_008.txt)...
✅ Selesai dalam 0.02 detik

📄 Memproses kasus PN0009 (case_009.txt)...
✅ Selesai dalam 0.00 detik

📄 Memproses kasus PN0010 (case_010.txt)...
✅ Selesai dalam 0.00 detik
⏱️ Progress: 10/30 file (10 sukses, 0 gagal)

📄 Memproses kasus PN0011 (case_011.txt)...
✅ Selesai dalam 0.01 detik

📄 Memproses kasus PN0012 (

TAHAP 3

In [31]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer, BertModel
import torch
from typing import List, Tuple
import os

# 1. Load processed data dengan penanganan error
def load_data(data_path: str) -> pd.DataFrame:
    try:
        # Baca file JSON
        with open(data_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # Pastikan format data sesuai
        if isinstance(data, list):
            return pd.DataFrame(data)
        elif isinstance(data, dict) and 'cases' in data:
            return pd.DataFrame(data['cases'])
        else:
            print("Format data tidak dikenali. Membuat data dummy...")
            return create_dummy_data()

    except FileNotFoundError:
        print(f"File {data_path} tidak ditemukan. Membuat data dummy...")
        return create_dummy_data()
    except json.JSONDecodeError:
        print(f"Error decoding JSON file. Membuat data dummy...")
        return create_dummy_data()

# Fungsi untuk membuat data dummy yang sesuai dengan output tahap 2
def create_dummy_data():
    dummy_cases = []
    for i in range(100):
        case_id = f"DUMMY{i+1:03d}"
        text_full = f"Ini adalah teks kasus dummy {i+1} untuk demonstrasi sistem."

        # Buat struktur yang sama dengan output tahap 2
        dummy_case = {
            "case_id": case_id,
            "file_name": f"dummy_{i+1}.txt",
            "no_perkara": f"123/Pid.B/2023/PN.Dummy.{i+1}",
            "tanggal": "2023-01-01",
            "jenis_perkara": ["Pidana", "Perdata", "TUN"][i % 3],
            "pasal": "Pasal 123 KUHP" if i % 3 == 0 else "Pasal 321 KUHP",
            "pihak": f"Penggugat: Dummy {i+1}; Tergugat: Dummy Defendant",
            "ringkasan_fakta": f"Ringkasan fakta dummy {i+1}",
            "argumen_hukum": f"Argumen hukum dummy {i+1}",
            "length_words": 500 + i,
            "length_chars": 2500 + i*10,
            "avg_word_length": 5.0 + i/100,
            "processing_time_sec": 0.5,
            "text_md5": "dummy_md5_hash",
            "text_full": text_full
        }
        dummy_cases.append(dummy_case)

    return pd.DataFrame(dummy_cases)

# 2. Splitting data
def split_data(df: pd.DataFrame, test_size: float = 0.2, random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state
    )
    return train_df, test_df

# 3. Vector Representation
class Vectorizer:
    def __init__(self, method: str = 'tfidf'):
        self.method = method
        self.vectorizer = None
        self.bert_model = None
        self.bert_tokenizer = None

    def fit(self, documents: List[str]):
        if self.method == 'tfidf':
            self.vectorizer = TfidfVectorizer()
            self.vectorizer.fit(documents)
        elif self.method == 'bert':
            model_name = 'indobenchmark/indobert-base-p1'
            self.bert_tokenizer = BertTokenizer.from_pretrained(model_name)
            self.bert_model = BertModel.from_pretrained(model_name)
            self.bert_model.eval()

    def transform(self, texts: pd.Series) -> np.ndarray:
        if self.method == 'tfidf':
            return self.vectorizer.transform(texts).toarray()
        elif self.method == 'bert':
            inputs = self.bert_tokenizer(
                texts.tolist(),
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors='pt'
            )
            with torch.no_grad():
                outputs = self.bert_model(**inputs)
            # Use [CLS] token embedding as document representation
            return outputs.last_hidden_state[:, 0, :].cpu().numpy()

# 4. Retrieval Function
class CaseRetriever:
    def __init__(self, vectorizer: Vectorizer, case_ids: List[str], vectors: np.ndarray):
        self.vectorizer = vectorizer
        self.case_ids = case_ids
        self.vectors = vectors

    def retrieve(self, query: str, k: int = 5) -> List[str]:
        # Preprocess and vectorize query
        query_vec = self.vectorizer.transform(pd.Series([query]))

        if self.vectorizer.method == 'tfidf':
            # Compute cosine similarity for TF-IDF
            similarities = cosine_similarity(query_vec, self.vectors).flatten()
        else:
            # Compute cosine similarity for BERT embeddings
            # Normalize vectors
            query_vec_norm = query_vec / np.linalg.norm(query_vec, axis=1, keepdims=True)
            vectors_norm = self.vectors / np.linalg.norm(self.vectors, axis=1, keepdims=True)
            similarities = np.dot(vectors_norm, query_vec_norm.T).flatten()

        # Get top-k indices
        top_indices = np.argsort(similarities)[::-1][:k]
        return [self.case_ids[i] for i in top_indices]

# 5. Main execution
if __name__ == "__main__":
    # Configurations
    BASE_DIR = "/content/drive/MyDrive/TGS PK/data"
    PROCESSED_DIR = os.path.join(BASE_DIR, "processed")  # Perbaiki path
    EVAL_DIR = os.path.join(BASE_DIR, "eval")  # Perbaiki path
    DATA_PATH = os.path.join(PROCESSED_DIR, "cases.json")
    RETRIEVAL_METHOD = 'tfidf'
    TEST_SIZE = 0.2
    TOP_K = 5

    # Ensure directories exist
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    os.makedirs(EVAL_DIR, exist_ok=True)

    # Load or create data
    df = load_data(DATA_PATH)
    print(f"Loaded {len(df)} cases")

    # Save dummy data if created
    if not os.path.exists(DATA_PATH):
        df.to_json(DATA_PATH, orient='records', indent=2, force_ascii=False)
        print(f"Data dummy disimpan di {DATA_PATH}")

    # Split data
    train_df, test_df = split_data(df, test_size=TEST_SIZE)
    print(f"Split data: {len(train_df)} training, {len(test_df)} test")

    # Initialize and fit vectorizer
    vectorizer = Vectorizer(method=RETRIEVAL_METHOD)
    vectorizer.fit(train_df['text_full'].tolist())
    print(f"Vectorizer initialized with method: {RETRIEVAL_METHOD}")

    # Transform training data
    train_vectors = vectorizer.transform(train_df['text_full'])
    print(f"Vectorized training data shape: {train_vectors.shape}")

    # Initialize retriever
    retriever = CaseRetriever(
        vectorizer=vectorizer,
        case_ids=train_df['case_id'].tolist(),
        vectors=train_vectors
    )
    print("CaseRetriever initialized")

    # 6. Create 10 sample queries for evaluation
    sample_queries = [
        {
            "query_id": "Q1",
            "query_text": "Kasus pencurian dengan kekerasan oleh kelompok bersenjata",
            "relevant_cases": ["DUMMY001", "DUMMY002"]  # Dummy case IDs
        },
        {
            "query_id": "Q2",
            "query_text": "Perkara perdata tentang wanprestasi dalam kontrak konstruksi",
            "relevant_cases": ["DUMMY003", "DUMMY004"]
        },
        {
            "query_id": "Q3",
            "query_text": "Pidana narkotika jenis sabu dengan jumlah besar",
            "relevant_cases": ["DUMMY005", "DUMMY006"]
        },
        {
            "query_id": "Q4",
            "query_text": "Sengketa tanah warisan antar saudara",
            "relevant_cases": ["DUMMY007", "DUMMY008"]
        },
        {
            "query_id": "Q5",
            "query_text": "Penganiayaan berat yang mengakibatkan kematian",
            "relevant_cases": ["DUMMY009", "DUMMY010"]
        },
        {
            "query_id": "Q6",
            "query_text": "Perbuatan melawan hukum oleh perusahaan terhadap konsumen",
            "relevant_cases": ["DUMMY011", "DUMMY012"]
        },
        {
            "query_id": "Q7",
            "query_text": "Pencemaran nama baik melalui media sosial",
            "relevant_cases": ["DUMMY013", "DUMMY014"]
        },
        {
            "query_id": "Q8",
            "query_text": "Pemalsuan dokumen untuk pengajuan kredit bank",
            "relevant_cases": ["DUMMY015", "DUMMY016"]
        },
        {
            "query_id": "Q9",
            "query_text": "Sengketa hak asuh anak setelah perceraian",
            "relevant_cases": ["DUMMY017", "DUMMY018"]
        },
        {
            "query_id": "Q10",
            "query_text": "Korupsi pengadaan barang di instansi pemerintah",
            "relevant_cases": ["DUMMY019", "DUMMY020"]
        }
    ]

    # Save queries to file
    queries_path = os.path.join(EVAL_DIR, "queries.json")
    with open(queries_path, 'w', encoding='utf-8') as f:
        json.dump(sample_queries, f, ensure_ascii=False, indent=2)
    print(f"Saved evaluation queries to {queries_path}")

    # 7. Test retrieval function for all 10 queries
    print("\nTesting retrieval function for 10 queries:")
    results_summary = {}

    for query in sample_queries:
        results = retriever.retrieve(query['query_text'], k=TOP_K)
        results_summary[query['query_id']] = results

        print(f"\nQuery {query['query_id']}: {query['query_text']}")
        print(f"Top {TOP_K} results: {results}")

    # Save results to file
    results_path = os.path.join(EVAL_DIR, "retrieval_results.json")
    with open(results_path, 'w', encoding='utf-8') as f:
        json.dump(results_summary, f, ensure_ascii=False, indent=2)
    print(f"\nSaved retrieval results to {results_path}")

    print("\nScript berhasil dijalankan!")

Loaded 30 cases
Split data: 24 training, 6 test
Vectorizer initialized with method: tfidf
Vectorized training data shape: (24, 10)
CaseRetriever initialized
Saved evaluation queries to /content/drive/MyDrive/TGS PK/data/eval/queries.json

Testing retrieval function for 10 queries:

Query Q1: Kasus pencurian dengan kekerasan oleh kelompok bersenjata
Top 5 results: ['PN0007', 'PN0020', 'PN0015', 'PN0011', 'PN0008']

Query Q2: Perkara perdata tentang wanprestasi dalam kontrak konstruksi
Top 5 results: ['PN0007', 'PN0020', 'PN0015', 'PN0011', 'PN0008']

Query Q3: Pidana narkotika jenis sabu dengan jumlah besar
Top 5 results: ['PN0007', 'PN0020', 'PN0015', 'PN0011', 'PN0008']

Query Q4: Sengketa tanah warisan antar saudara
Top 5 results: ['PN0007', 'PN0020', 'PN0015', 'PN0011', 'PN0008']

Query Q5: Penganiayaan berat yang mengakibatkan kematian
Top 5 results: ['PN0007', 'PN0020', 'PN0015', 'PN0011', 'PN0008']

Query Q6: Perbuatan melawan hukum oleh perusahaan terhadap konsumen
Top 5 results

TAHAP 4

In [32]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer, BertModel
import torch
from typing import List, Tuple, Dict
import os
from collections import Counter

# 1. Load processed data dengan penanganan error
def load_data(data_path: str) -> pd.DataFrame:
    try:
        with open(data_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        df = pd.DataFrame(data)

        # Pastikan kolom 'solution' ada
        if 'solution' not in df.columns:
            print("Kolom 'solution' tidak ditemukan. Membuat solusi dummy...")
            # Coba gunakan kolom 'jenis_perkara' sebagai dasar
            if 'jenis_perkara' in df.columns:
                df['solution'] = df['jenis_perkara'].apply(
                    lambda x: f"Solusi untuk kasus {x}"
                )
            else:
                df['solution'] = "Solusi tidak tersedia"
        return df
    except FileNotFoundError:
        print(f"File {data_path} tidak ditemukan. Membuat data dummy...")
        return create_dummy_data()
    except json.JSONDecodeError:
        print(f"Error decoding JSON. Membuat data dummy...")
        return create_dummy_data()

# Fungsi untuk membuat data dummy dengan solusi yang lebih realistis
def create_dummy_data():
    # Daftar kata untuk membuat variasi teks
    kata_umum = ["pengadilan", "hakim", "terdakwa", "penggugat", "tergugat", "majelis",
                "sidang", "pembuktian", "saksi", "barang", "bukti", "putusan", "hukum",
                "pidana", "perdata", "tata usaha negara", "gugatan", "tuntutan", "eksepsi",
                "mengajukan", "pemeriksaan", "pengadilan negeri", "penuntut umum", "jaksa",
                "penasehat hukum", "kuasa hukum", "mohon", "memutuskan", "mempertimbangkan"]

    kata_khusus = {
        "Pidana": ["pencurian", "pembunuhan", "narkotika", "penganiayaan", "korupsi",
                  "penipuan", "pemerasan", "penadahan", "pencabulan", "perampokan"],
        "Perdata": ["wanprestasi", "perjanjian", "ganti rugi", "sengketa", "kontrak",
                   "utang piutang", "jaminan", "fidusia", "hak milik", "sita"],
        "TUN": ["izin", "pencabutan", "instansi", "keputusan", "administrasi",
                "pemerintahan", "pengadaan", "tender", "pelayanan publik", "reklamasi"]
    }

    solutions = {
        "Pidana": [
            "Menghukum terdakwa dengan pidana penjara selama {durasi} tahun",
            "Menghukum terdakwa membayar denda sebesar Rp {jumlah}",
            "Menghukum terdakwa dengan pidana penjara selama {durasi} tahun dan denda Rp {jumlah}",
            "Menyatakan terdakwa terbukti bersalah melakukan tindak pidana {pasal}",
            "Menghukum terdakwa untuk mengganti kerugian sebesar Rp {jumlah} kepada korban"
        ],
        "Perdata": [
            "Menghukum tergugat membayar ganti rugi sebesar Rp {jumlah}",
            "Membatalkan perjanjian yang menjadi objek sengketa",
            "Menghukum tergugat memenuhi kewajiban kontrak",
            "Menetapkan bahwa objek sengketa adalah milik penggugat",
            "Memerintahkan pengosongan objek sengketa dalam jangka waktu {waktu}"
        ],
        "TUN": [
            "Membatalkan keputusan administrasi yang dikeluarkan instansi",
            "Memerintahkan instansi mengeluarkan izin yang dimohonkan",
            "Menolak permohonan pencabutan izin usaha",
            "Memerintahkan instansi untuk mengembalikan dokumen yang disita",
            "Menghukum instansi membayar ganti rugi sebesar Rp {jumlah}"
        ]
    }

    dummy_cases = []
    for i in range(100):
        case_id = f"PN{i+1:04d}"
        category = ["Pidana", "Perdata", "TUN"][i % 3]

        # Buat teks lebih bervariasi dan realistis
        num_paragraphs = np.random.randint(3, 8)
        paragraphs = []

        for _ in range(num_paragraphs):
            sentence_count = np.random.randint(3, 8)
            sentences = []

            for __ in range(sentence_count):
                word_count = np.random.randint(8, 20)
                # Campur kata umum dan kata khusus
                word_pool = kata_umum + kata_khusus[category]
                sentence_words = np.random.choice(word_pool, word_count)
                sentence = " ".join(sentence_words).capitalize() + "."
                sentences.append(sentence)

            paragraphs.append(" ".join(sentences))

        text_full = "\n\n".join(paragraphs)

        # Pilih solusi secara acak berdasarkan kategori
        solution_template = np.random.choice(solutions[category])
        solution = solution_template.format(
            durasi=np.random.randint(1, 10),
            jumlah=np.random.randint(1000000, 100000000, step=1000000),
            waktu=f"{np.random.randint(1, 12)} bulan",
            pasal=f"{np.random.randint(100,500)} KUHP"
        )

        dummy_cases.append({
            "case_id": case_id,
            "text_full": text_full,
            "solution": solution,
            "jenis_perkara": category,
            "no_perkara": f"{np.random.randint(100,999)}/PID/{i+1}/2023",
            "tanggal": f"2023-{np.random.randint(1,12):02d}-{np.random.randint(1,28):02d}",
            "pasal": f"Pasal {np.random.randint(100,500)} KUHP",
            "pihak": f"Penggugat: Pihak {i+1}; Tergugat: Pihak {i+2}",
            "ringkasan_fakta": f"Ringkasan fakta untuk kasus {case_id}",
            "argumen_hukum": f"Argumen hukum untuk kasus {case_id}",
            "length_words": len(text_full.split()),
            "length_chars": len(text_full),
            "avg_word_length": len(text_full) / max(1, len(text_full.split())),
            "processing_time_sec": 0.5,
            "text_md5": f"dummy_md5_{i}",
        })
    return pd.DataFrame(dummy_cases)

# 2. Splitting data
def split_data(df: pd.DataFrame, test_size: float = 0.2, random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state
    )
    return train_df, test_df

# 3. Vector Representation
class Vectorizer:
    def __init__(self, method: str = 'tfidf'):
        self.method = method
        self.vectorizer = None
        self.bert_model = None
        self.bert_tokenizer = None

    def fit(self, documents: List[str]):
        if self.method == 'tfidf':
            self.vectorizer = TfidfVectorizer()
            self.vectorizer.fit(documents)
        elif self.method == 'bert':
            model_name = 'indobenchmark/indobert-base-p1'
            self.bert_tokenizer = BertTokenizer.from_pretrained(model_name)
            self.bert_model = BertModel.from_pretrained(model_name)
            self.bert_model.eval()

    def transform(self, texts: pd.Series) -> np.ndarray:
        if texts.empty:
            return np.array([])

        if self.method == 'tfidf':
            return self.vectorizer.transform(texts).toarray()
        elif self.method == 'bert':
            # Tokenisasi batch untuk efisiensi
            inputs = self.bert_tokenizer(
                texts.tolist(),
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors='pt'
            )

            with torch.no_grad():
                outputs = self.bert_model(**inputs)

            # Gunakan embedding token [CLS] sebagai representasi dokumen
            return outputs.last_hidden_state[:, 0, :].cpu().numpy()

# 4. Retrieval Function dengan kembalian skor dan penanganan vektor nol
class CaseRetriever:
    def __init__(self, vectorizer: Vectorizer, case_ids: List[str], vectors: np.ndarray):
        self.vectorizer = vectorizer
        self.case_ids = case_ids
        self.vectors = vectors

        # Precompute norms untuk efisiensi dan hindari pembagian nol
        self.vector_norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        # Ganti nilai 0 dengan epsilon kecil untuk hindari pembagian nol
        self.vector_norms[self.vector_norms == 0] = 1e-10

    def retrieve_with_scores(self, query: str, k: int = 5) -> Tuple[List[str], List[float]]:
        if len(self.vectors) == 0:
            return [], []

        # Preprocess dan vectorize query
        query_vec = self.vectorizer.transform(pd.Series([query]))

        if query_vec.size == 0 or np.all(query_vec == 0):
            return [], []

        if self.vectorizer.method == 'tfidf':
            # Hitung cosine similarity langsung
            similarities = cosine_similarity(query_vec, self.vectors).flatten()
        else:
            # Normalisasi vektor query
            query_norm = np.linalg.norm(query_vec, axis=1, keepdims=True)
            query_norm[query_norm == 0] = 1e-10  # Hindari pembagian nol
            query_vec_norm = query_vec / query_norm

            # Normalisasi vektor database
            vectors_norm = self.vectors / self.vector_norms

            # Hitung cosine similarity
            similarities = np.dot(vectors_norm, query_vec_norm.T).flatten()

            # Tangani kemungkinan nilai NaN
            similarities = np.nan_to_num(similarities, nan=0.0)

            # Pastikan skor dalam rentang yang valid
            similarities = np.clip(similarities, -1.0, 1.0)

        # Dapatkan indeks top-k
        top_indices = np.argsort(similarities)[::-1][:k]
        return (
            [self.case_ids[i] for i in top_indices],
            similarities[top_indices].tolist()
        )

# 5. Fungsi untuk memprediksi solusi
def predict_outcome(query: str, retriever: CaseRetriever, case_solutions: Dict[str, str], k: int = 5, method: str = 'weighted') -> str:
    """
    Memprediksi solusi berdasarkan kasus serupa

    Args:
        query: Teks query kasus baru
        retriever: Objek untuk retrieval kasus
        case_solutions: Mapping case_id ke solusi
        k: Jumlah kasus teratas yang dipertimbangkan
        method: Metode prediksi ('majority' atau 'weighted')

    Returns:
        Predicted solution text
    """
    # Dapatkan kasus serupa dengan skor
    top_k_case_ids, scores = retriever.retrieve_with_scores(query, k=k)

    # Jika tidak ada hasil, kembalikan pesan default
    if not top_k_case_ids:
        return "Tidak ada kasus serupa yang ditemukan"

    # Dapatkan solusi untuk kasus teratas
    solutions = []
    for case_id in top_k_case_ids:
        if case_id in case_solutions:
            solutions.append(case_solutions[case_id])
        else:
            solutions.append("Solusi tidak tersedia")

    if method == 'majority':
        # Majority vote: pilih solusi yang paling sering muncul
        solution_counter = Counter(solutions)
        most_common = solution_counter.most_common(1)
        return most_common[0][0] if most_common else "Tidak ada solusi yang ditemukan"

    elif method == 'weighted':
        # Weighted similarity: solusi dengan bobot skor tertinggi
        solution_scores = {}
        for solution, score in zip(solutions, scores):
            if solution not in solution_scores:
                solution_scores[solution] = 0
            solution_scores[solution] += max(0, score)  # Hanya gunakan skor positif

        # Pilih solusi dengan total skor tertinggi
        if solution_scores:
            return max(solution_scores.items(), key=lambda x: x[1])[0]
        else:
            return "Tidak ada solusi yang ditemukan"

    else:
        raise ValueError(f"Metode tidak valid: {method}")

# 6. Main execution
if __name__ == "__main__":
    # Konfigurasi path yang benar
    BASE_DIR = "/content/drive/MyDrive/TGS PK/data"
    PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
    EVAL_DIR = os.path.join(BASE_DIR, "eval")
    RESULTS_DIR = os.path.join(BASE_DIR, "results")

    # Path file
    DATA_PATH = os.path.join(PROCESSED_DIR, "cases.json")
    QUERIES_PATH = os.path.join(EVAL_DIR, "queries.json")
    PREDICTIONS_JSON_PATH = os.path.join(RESULTS_DIR, "predictions.json")
    PREDICTIONS_CSV_PATH = os.path.join(RESULTS_DIR, "predictions_flat.csv")

    RETRIEVAL_METHOD = 'tfidf'  # tfidf atau bert
    TEST_SIZE = 0.2
    TOP_K = 5

    # Pastikan direktori ada
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    os.makedirs(EVAL_DIR, exist_ok=True)
    os.makedirs(RESULTS_DIR, exist_ok=True)

    # Muat data
    print(f"Muat data dari: {DATA_PATH}")
    df = load_data(DATA_PATH)

    # Jika tidak ada data, buat dan simpan data dummy
    if df.empty:
        print("Data kosong, membuat data dummy...")
        df = create_dummy_data()
        df.to_json(DATA_PATH, orient='records', indent=2, force_ascii=False)
        print(f"Data dummy disimpan di {DATA_PATH}")
    else:
        print(f"Data berhasil dimuat: {len(df)} baris")

    # Split data
    print("Split data menjadi training dan test set...")
    train_df, test_df = split_data(df, test_size=TEST_SIZE)

    # Pastikan kita memiliki teks untuk diproses
    if train_df.empty or 'text_full' not in train_df.columns:
        print("Error: Data training tidak valid. Keluar...")
        exit()

    print(f"Jumlah data training: {len(train_df)}")
    print(f"Jumlah data test: {len(test_df)}")

    # Inisialisasi dan fit vectorizer
    print(f"Inisialisasi vectorizer dengan metode: {RETRIEVAL_METHOD}")
    vectorizer = Vectorizer(method=RETRIEVAL_METHOD)
    vectorizer.fit(train_df['text_full'])

    # Transform data training
    print("Transform data training...")
    train_vectors = vectorizer.transform(train_df['text_full'])

    # Inisialisasi retriever dengan penanganan vektor nol
    retriever = CaseRetriever(
        vectorizer=vectorizer,
        case_ids=train_df['case_id'].tolist(),
        vectors=train_vectors
    )

    # Buat mapping case_id ke solusi
    case_solutions = dict(zip(train_df['case_id'], train_df['solution']))

    # Load sample queries for evaluation
    if os.path.exists(QUERIES_PATH):
        with open(QUERIES_PATH, 'r', encoding='utf-8') as f:
            sample_queries = json.load(f)
        print(f"Loaded evaluation queries from {QUERIES_PATH}")
    else:
        # Create 10 sample queries for evaluation
        sample_queries = [
            {
                "query_id": "Q1",
                "query_text": "Kasus pencurian dengan kekerasan oleh kelompok bersenjata",
                "relevant_cases": ["PN0001", "PN0002"]
            },
            {
                "query_id": "Q2",
                "query_text": "Perkara perdata tentang wanprestasi dalam kontrak konstruksi",
                "relevant_cases": ["PN0003", "PN0004"]
            },
            {
                "query_id": "Q3",
                "query_text": "Pidana narkotika jenis sabu dengan jumlah besar",
                "relevant_cases": ["PN0005", "PN0006"]
            },
            {
                "query_id": "Q4",
                "query_text": "Sengketa tanah warisan antar saudara",
                "relevant_cases": ["PN0007", "PN0008"]
            },
            {
                "query_id": "Q5",
                "query_text": "Penganiayaan berat yang mengakibatkan kematian",
                "relevant_cases": ["PN0009", "PN0010"]
            },
            {
                "query_id": "Q6",
                "query_text": "Perbuatan melawan hukum oleh perusahaan terhadap konsumen",
                "relevant_cases": ["PN0011", "PN0012"]
            },
            {
                "query_id": "Q7",
                "query_text": "Pencemaran nama baik melalui media sosial",
                "relevant_cases": ["PN0013", "PN0014"]
            },
            {
                "query_id": "Q8",
                "query_text": "Pemalsuan dokumen untuk pengajuan kredit bank",
                "relevant_cases": ["PN0015", "PN0016"]
            },
            {
                "query_id": "Q9",
                "query_text": "Sengketa hak asuh anak setelah perceraian",
                "relevant_cases": ["PN0017", "PN0018"]
            },
            {
                "query_id": "Q10",
                "query_text": "Korupsi pengadaan barang di instansi pemerintah",
                "relevant_cases": ["PN0019", "PN0020"]
            }
        ]
        # Save queries to file
        with open(QUERIES_PATH, 'w', encoding='utf-8') as f:
            json.dump(sample_queries, f, ensure_ascii=False, indent=2)
        print(f"Created evaluation queries at {QUERIES_PATH}")

    # 7. Demo manual: prediksi solusi untuk setiap query
    predictions = []

    print("\nDemo Manual Prediksi Solusi:")
    for query in sample_queries:
        # Prediksi dengan majority vote
        majority_solution = predict_outcome(
            query['query_text'],
            retriever,
            case_solutions,
            k=TOP_K,
            method='majority'
        )

        # Prediksi dengan weighted similarity
        weighted_solution = predict_outcome(
            query['query_text'],
            retriever,
            case_solutions,
            k=TOP_K,
            method='weighted'
        )

        # Dapatkan top-k case ids dan skor
        top_k_case_ids, scores = retriever.retrieve_with_scores(query['query_text'], k=TOP_K)

        # Dapatkan detail kasus
        top_k_details = []
        for case_id, score in zip(top_k_case_ids, scores):
            case_row = train_df[train_df['case_id'] == case_id]
            if not case_row.empty:
                case_data = case_row.iloc[0]
                top_k_details.append({
                    "case_id": case_id,
                    "solution": case_data['solution'],
                    "jenis_perkara": case_data.get('jenis_perkara', 'N/A'),
                    "similarity_score": score
                })

        # Simpan hasil untuk JSON
        predictions.append({
            "query_id": query['query_id'],
            "query_text": query['query_text'],
            "predicted_solution_majority": majority_solution,
            "predicted_solution_weighted": weighted_solution,
            "top_k_cases": top_k_details
        })

        # Tampilkan hasil demo
        print(f"\n{'='*80}")
        print(f"QUERY {query['query_id']}: {query['query_text']}")
        print(f"\nSolusi (majority vote): {majority_solution}")
        print(f"Solusi (weighted similarity): {weighted_solution}")

        print("\nTop kasus yang ditemukan:")
        for i, case_detail in enumerate(top_k_details, 1):
            print(f"  {i}. {case_detail['case_id']} - Skor: {case_detail['similarity_score']:.4f}")
            print(f"     Jenis: {case_detail['jenis_perkara']}")
            print(f"     Solusi: {case_detail['solution'][:100]}...")

    # 8. Simpan hasil prediksi ke JSON (karena struktur nested)
    with open(PREDICTIONS_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)
    print(f"\nHasil prediksi lengkap disimpan di {PREDICTIONS_JSON_PATH}")

    # Simpan versi flat untuk CSV
    flat_predictions = []
    for pred in predictions:
        flat_pred = {
            "query_id": pred["query_id"],
            "query_text": pred["query_text"],
            "predicted_solution_majority": pred["predicted_solution_majority"],
            "predicted_solution_weighted": pred["predicted_solution_weighted"]
        }

        # Tambahkan top-k cases sebagai string
        top_cases_str = []
        for case in pred["top_k_cases"]:
            top_cases_str.append(f"{case['case_id']} ({case['similarity_score']:.4f})")
        flat_pred["top_k_cases"] = "; ".join(top_cases_str)

        flat_predictions.append(flat_pred)

    # Simpan flat predictions ke CSV
    predictions_df = pd.DataFrame(flat_predictions)
    predictions_df.to_csv(PREDICTIONS_CSV_PATH, index=False)
    print(f"Versi flat hasil prediksi disimpan di {PREDICTIONS_CSV_PATH}")

    print("\nTahap 4 - Solution Reuse selesai!")

Muat data dari: /content/drive/MyDrive/TGS PK/data/processed/cases.json
Kolom 'solution' tidak ditemukan. Membuat solusi dummy...
Data berhasil dimuat: 30 baris
Split data menjadi training dan test set...
Jumlah data training: 24
Jumlah data test: 6
Inisialisasi vectorizer dengan metode: tfidf
Transform data training...
Loaded evaluation queries from /content/drive/MyDrive/TGS PK/data/eval/queries.json

Demo Manual Prediksi Solusi:

QUERY Q1: Kasus pencurian dengan kekerasan oleh kelompok bersenjata

Solusi (majority vote): Tidak ada kasus serupa yang ditemukan
Solusi (weighted similarity): Tidak ada kasus serupa yang ditemukan

Top kasus yang ditemukan:

QUERY Q2: Perkara perdata tentang wanprestasi dalam kontrak konstruksi

Solusi (majority vote): Tidak ada kasus serupa yang ditemukan
Solusi (weighted similarity): Tidak ada kasus serupa yang ditemukan

Top kasus yang ditemukan:

QUERY Q3: Pidana narkotika jenis sabu dengan jumlah besar

Solusi (majority vote): Tidak ada kasus serupa 

TAHAP 5

In [33]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer, BertModel
import torch
from typing import List, Tuple, Dict, Union
import os
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import re
from faker import Faker
import random

# Inisialisasi Faker untuk teks dummy
fake = Faker('id_ID')

# 1. Load processed data dengan penanganan error
def load_data(data_path: str) -> pd.DataFrame:
    try:
        with open(data_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        df = pd.DataFrame(data)

        # Pastikan kolom 'solution' ada
        if 'solution' not in df.columns:
            print("Kolom 'solution' tidak ditemukan. Membuat solusi dummy...")

            # Jika ada kolom 'argumen_hukum', gunakan sebagai solution
            if 'argumen_hukum' in df.columns:
                df['solution'] = df['argumen_hukum']
                print("Menggunakan 'argumen_hukum' sebagai solution")
            else:
                # Buat solusi dummy berdasarkan jenis_perkara
                def create_dummy_solution(row):
                    if 'jenis_perkara' in row:
                        if "Pidana" in row['jenis_perkara']:
                            return f"Menghukum terdakwa berdasarkan putusan {fake.city()}"
                        elif "Perdata" in row['jenis_perkara']:
                            return f"Menghukum tergugat membayar ganti rugi sebesar {fake.random_int(min=10000000, max=500000000)}"
                        elif "TUN" in row['jenis_perkara']:
                            return f"Membatalkan keputusan administrasi oleh instansi terkait"
                    return "Solusi dummy"

                df['solution'] = df.apply(create_dummy_solution, axis=1)
                print("Membuat solusi dummy berdasarkan jenis_perkara")

        return df
    except FileNotFoundError:
        print(f"File {data_path} tidak ditemukan. Membuat data dummy...")
        return create_dummy_data()
    except json.JSONDecodeError:
        print(f"Error decoding JSON. Membuat data dummy...")
        return create_dummy_data()

# Fungsi untuk membuat data dummy dengan solusi yang lebih realistis
def create_dummy_data():
    # Detail kasus yang lebih spesifik untuk setiap kategori
    pidana_details = [
        "pencurian dengan kekerasan oleh kelompok bersenjata",
        "penganiayaan berat mengakibatkan kematian",
        "narkotika jenis sabu dengan jumlah besar",
        "pemalsuan dokumen untuk pengajuan kredit bank",
        "korupsi pengadaan barang di instansi pemerintah"
    ]

    perdata_details = [
        "wanprestasi dalam kontrak konstruksi",
        "sengketa tanah warisan antar saudara",
        "perbuatan melawan hukum oleh perusahaan terhadap konsumen",
        "sengketa hak asuh anak setelah perceraian",
        "gugatan ganti rugi atas cacat produk"
    ]

    tun_details = [
        "pencabutan izin usaha tanpa pemberitahuan",
        "penolakan permohonan izin mendirikan bangunan",
        "pemberhentian tidak hormat pegawai negeri",
        "pembatalan sertifikat tanah",
        "penetapan pajak yang tidak wajar"
    ]

    solutions = {
        "Pidana": [
            "Menghukum terdakwa dengan pidana penjara selama 5 tahun",
            "Menghukum terdakwa dengan pidana penjara selama 3 tahun",
            "Menghukum terdakwa dengan pidana penjara selama 10 tahun",
            "Menghukum terdakwa membayar denda sebesar Rp 50.000.000",
            "Menyita barang bukti terkait kasus narkotika"
        ],
        "Perdata": [
            "Menghukum tergugat membayar ganti rugi sebesar Rp 100.000.000",
            "Membatalkan perjanjian yang menjadi objek sengketa",
            "Menghukum tergugat memenuhi kewajiban kontrak",
            "Memerintahkan pemindahan hak atas tanah",
            "Menetapkan hak asuh anak kepada penggugat"
        ],
        "TUN": [
            "Membatalkan keputusan administrasi yang dikeluarkan instansi",
            "Memerintahkan instansi mengeluarkan izin yang dimohonkan",
            "Menolak permohonan pencabutan izin usaha",
            "Memerintahkan pemulihan status kepegawaian",
            "Membatalkan penetapan pajak yang diterbitkan"
        ]
    }

    dummy_cases = []
    for i in range(100):
        case_id = f"DUMMY{i+1:03d}"
        category = ["Pidana", "Perdata", "TUN"][i % 3]

        # Pilih detail berdasarkan kategori
        if category == "Pidana":
            detail = random.choice(pidana_details)
            solution = random.choice(solutions["Pidana"])
        elif category == "Perdata":
            detail = random.choice(perdata_details)
            solution = random.choice(solutions["Perdata"])
        else:
            detail = random.choice(tun_details)
            solution = random.choice(solutions["TUN"])

        # Buat teks kasus yang lebih panjang dan realistis
        text_full = (
            f"Putusan {category} No. {case_id}. "
            f"Perkara ini terkait dengan kasus {detail}. "
            f"{fake.paragraph(nb_sentences=5)} "
            f"Setelah mempertimbangkan bukti-bukti yang diajukan, "
            f"hakim memutuskan {solution.lower()}."
        )

        # Tambahkan keyword terkait ke dalam teks untuk memastikan relevansi
        keywords = detail.split()[:2]
        for keyword in keywords:
            text_full += f" {keyword} " * random.randint(1, 3)

        dummy_cases.append({
            "case_id": case_id,
            "text_full": text_full,
            "jenis_perkara": category,
            "solution": solution,
            "metadata": {
                "court": "Pengadilan Negeri " + fake.city(),
                "date": fake.date_between(start_date='-5y', end_date='today').strftime('%Y-%m-%d'),
                "category": category
            }
        })

    return pd.DataFrame(dummy_cases)

# 2. Splitting data - TIDAK DIGUNAKAN untuk retriever, hanya untuk evaluasi
def split_data(df: pd.DataFrame, test_size: float = 0.2, random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # Hanya untuk evaluasi prediksi solusi
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state
    )
    return train_df, test_df

# 3. Vector Representation
class Vectorizer:
    def __init__(self, method: str = 'tfidf'):
        self.method = method
        self.vectorizer = None
        self.bert_model = None
        self.bert_tokenizer = None

    def fit(self, documents: List[str]):
        if self.method == 'tfidf':
            self.vectorizer = TfidfVectorizer(max_features=10000)
            self.vectorizer.fit(documents)
        elif self.method == 'bert':
            model_name = 'indobenchmark/indobert-base-p1'
            self.bert_tokenizer = BertTokenizer.from_pretrained(model_name)
            self.bert_model = BertModel.from_pretrained(model_name)
            self.bert_model.eval()

    def transform(self, texts: pd.Series) -> np.ndarray:
        if texts.empty:
            return np.array([])

        if self.method == 'tfidf':
            return self.vectorizer.transform(texts).toarray()
        elif self.method == 'bert':
            # Tokenisasi batch untuk efisiensi
            inputs = self.bert_tokenizer(
                texts.tolist(),
                padding=True,
                truncation=True,
                max_length=256,  # Kurangi max_length untuk efisiensi
                return_tensors='pt'
            )

            with torch.no_grad():
                outputs = self.bert_model(**inputs)

            # Gunakan embedding token [CLS] sebagai representasi dokumen
            return outputs.last_hidden_state[:, 0, :].cpu().numpy()

# 4. Retrieval Function dengan kembalian skor
class CaseRetriever:
    def __init__(self, vectorizer: Vectorizer, case_ids: List[str], vectors: np.ndarray):
        self.vectorizer = vectorizer
        self.case_ids = case_ids
        self.vectors = vectors

    def retrieve_with_scores(self, query: str, k: int = 5) -> Tuple[List[str], List[float]]:
        if len(self.vectors) == 0:
            return [], []

        # Preprocess dan vectorize query
        query_vec = self.vectorizer.transform(pd.Series([query]))

        if query_vec.size == 0:
            return [], []

        if self.vectorizer.method == 'tfidf':
            similarities = cosine_similarity(query_vec, self.vectors).flatten()
        else:
            # Normalisasi vektor untuk cosine similarity
            query_vec_norm = query_vec / np.linalg.norm(query_vec, axis=1, keepdims=True)
            vectors_norm = self.vectors / np.linalg.norm(self.vectors, axis=1, keepdims=True)
            similarities = np.dot(vectors_norm, query_vec_norm.T).flatten()

        # Dapatkan indeks top-k
        top_indices = np.argsort(similarities)[::-1][:k]
        return (
            [self.case_ids[i] for i in top_indices],
            similarities[top_indices].tolist()
        )

# 5. Fungsi untuk memprediksi solusi
def predict_outcome(query: str, retriever: CaseRetriever, case_solutions: Dict[str, str],
                   k: int = 5, method: str = 'weighted') -> str:
    top_k_case_ids, scores = retriever.retrieve_with_scores(query, k=k)

    if not top_k_case_ids:
        return "Tidak ada kasus serupa yang ditemukan"

    # Dapatkan solusi untuk kasus teratas
    solutions = []
    for case_id in top_k_case_ids:
        if case_id in case_solutions:
            solutions.append(case_solutions[case_id])
        else:
            solutions.append("Solusi tidak tersedia")

    if method == 'majority':
        # Majority vote: pilih solusi yang paling sering muncul
        solution_counter = Counter(solutions)
        most_common = solution_counter.most_common(1)
        return most_common[0][0] if most_common else "Tidak ada solusi"

    elif method == 'weighted':
        # Weighted similarity: solusi dengan bobot skor tertinggi
        solution_scores = {}
        for solution, score in zip(solutions, scores):
            if solution not in solution_scores:
                solution_scores[solution] = 0
            solution_scores[solution] += score

        if solution_scores:
            return max(solution_scores.items(), key=lambda x: x[1])[0]
        else:
            return "Tidak ada solusi"

    else:
        raise ValueError(f"Metode tidak valid: {method}")

# 6. Evaluasi Retrieval
def eval_retrieval(queries: List[Dict], retriever: CaseRetriever, k: int = 5) -> pd.DataFrame:
    results = []

    for query in queries:
        query_id = query['query_id']
        query_text = query['query_text']
        ground_truth = set(query['relevant_cases'])

        # Dapatkan hasil retrieval
        retrieved, _ = retriever.retrieve_with_scores(query_text, k=k)
        retrieved_set = set(retrieved)

        # Hitung metrik
        true_positives = len(retrieved_set & ground_truth)
        false_positives = len(retrieved_set - ground_truth)
        false_negatives = len(ground_truth - retrieved_set)

        # Hitung metrik dengan penanganan kasus khusus
        precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
        recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        accuracy = 1 if true_positives > 0 else 0  # Binary hit rate

        results.append({
            'query_id': query_id,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'accuracy': accuracy,
            'true_positives': true_positives,
            'false_positives': false_positives,
            'false_negatives': false_negatives,
            'retrieved': ', '.join(retrieved),
            'ground_truth': ', '.join(ground_truth)
        })

    return pd.DataFrame(results)

# 7. Evaluasi Prediksi
def eval_prediction(queries: List[Dict], retriever: CaseRetriever,
                   case_solutions: Dict[str, str], method: str = 'weighted') -> pd.DataFrame:
    results = []

    for query in queries:
        query_id = query['query_id']
        query_text = query['query_text']

        # Dapatkan solusi ground truth (majority dari kasus relevan)
        relevant_solutions = []
        for case_id in query['relevant_cases']:
            if case_id in case_solutions:
                relevant_solutions.append(case_solutions[case_id])

        if not relevant_solutions:
            ground_truth_solution = "Tidak ada solusi referensi"
        else:
            solution_counter = Counter(relevant_solutions)
            ground_truth_solution = solution_counter.most_common(1)[0][0]

        # Prediksi solusi
        predicted_solution = predict_outcome(
            query_text,
            retriever,
            case_solutions,
            k=5,
            method=method
        )

        # Hitung kesamaan (1 jika sama, 0 jika tidak)
        correct = 1 if predicted_solution == ground_truth_solution else 0

        results.append({
            'query_id': query_id,
            'predicted_solution': predicted_solution,
            'ground_truth_solution': ground_truth_solution,
            'correct': correct
        })

    return pd.DataFrame(results)

# 8. Visualisasi metrik retrieval
def visualize_retrieval_metrics(metrics_df: pd.DataFrame, method_name: str, output_dir: str):
    """Visualisasi metrik evaluasi retrieval"""
    # Pastikan direktori output ada
    os.makedirs(output_dir, exist_ok=True)

    # Buat plot
    plt.figure(figsize=(14, 8))

    # Buat plot metrik per query
    ax = metrics_df.plot(
        x='query_id',
        y=['precision', 'recall', 'f1', 'accuracy'],
        kind='bar',
        width=0.8,
        figsize=(14, 8))

    # Atur label dan judul
    plt.title(f'Metrik Evaluasi Retrieval per Query ({method_name.upper()})', fontsize=16)
    plt.xlabel('Query ID', fontsize=12)
    plt.ylabel('Nilai', fontsize=12)
    plt.ylim(0, 1.1)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    # Tambahkan nilai di atas setiap bar
    for p in ax.patches:
        height = p.get_height()
        if height > 0:  # Hanya tambahkan label untuk nilai > 0
            ax.annotate(
                f'{height:.2f}',
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='center',
                xytext=(0, 5),
                textcoords='offset points',
                fontsize=9
            )

    # Atur layout
    plt.tight_layout()

    # Simpan plot
    plot_path = os.path.join(output_dir, f'retrieval_metrics_{method_name}.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"Visualisasi metrik disimpan di: {plot_path}")
    return plot_path

# 9. Visualisasi perbandingan model
def visualize_model_comparison(summary_df: pd.DataFrame, output_dir: str):
    # Grafik perbandingan model
    plt.figure(figsize=(12, 8))
    ax = summary_df.set_index('method')[['precision', 'recall', 'f1', 'accuracy']].plot(kind='bar')
    plt.title('Perbandingan Metrik Evaluasi Retrieval antar Model', fontsize=14)
    plt.ylabel('Nilai', fontsize=12)
    plt.ylim(0, 1.0)
    plt.xticks(rotation=0)
    plt.legend(title='Metrik', fontsize=10)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()

    # Tambahkan nilai di atas setiap bar
    for p in ax.patches:
        height = p.get_height()
        ax.annotate(
            f'{height:.2f}',
            (p.get_x() + p.get_width() / 2., height),
            ha='center', va='center',
            xytext=(0, 5),
            textcoords='offset points',
            fontsize=9
        )

    # Simpan plot
    plot_path = os.path.join(output_dir, 'retrieval_model_comparison.png')
    plt.savefig(plot_path, dpi=300)
    plt.close()

    print(f"Visualisasi perbandingan model disimpan di: {plot_path}")

# 10. Visualisasi perbandingan metode prediksi
def visualize_prediction_comparison(pred_summary: pd.DataFrame, output_dir: str):
    # Grafik perbandingan metode prediksi
    plt.figure(figsize=(12, 8))
    ax = pred_summary.set_index(['retrieval_method', 'prediction_method'])['accuracy'].unstack().plot(kind='bar')
    plt.title('Perbandingan Akurasi Prediksi Berdasarkan Metode', fontsize=14)
    plt.ylabel('Akurasi', fontsize=12)
    plt.ylim(0, 1.0)
    plt.xticks(rotation=0)
    plt.legend(title='Metode Prediksi', fontsize=10)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()

    # Tambahkan nilai di atas setiap bar
    for p in ax.patches:
        height = p.get_height()
        ax.annotate(
            f'{height:.2f}',
            (p.get_x() + p.get_width() / 2., height),
            ha='center', va='center',
            xytext=(0, 5),
            textcoords='offset points',
            fontsize=9
        )

    # Simpan plot
    plot_path = os.path.join(output_dir, 'prediction_method_comparison.png')
    plt.savefig(plot_path, dpi=300)
    plt.close()

    print(f"Visualisasi perbandingan prediksi disimpan di: {plot_path}")

# 11. Analisis kegagalan retrieval
def analyze_retrieval_failures(metrics_df: pd.DataFrame, method_name: str, output_dir: str):
    # Identifikasi query dengan performa terburuk berdasarkan F1-score
    worst_queries = metrics_df.nsmallest(3, 'f1')

    # Simpan analisis dalam format JSON
    analysis = []
    for _, row in worst_queries.iterrows():
        analysis.append({
            'query_id': row['query_id'],
            'f1_score': row['f1'],
            'precision': row['precision'],
            'recall': row['recall'],
            'retrieved_cases': row['retrieved'],
            'ground_truth': row['ground_truth'],
            'potential_causes': [
                "Kosakata query berbeda dengan dokumen referensi",
                "Domain kasus tidak terwakili dalam data training",
                "Representasi vektor tidak menangkap semantik kasus",
                "Kasus baru memiliki karakteristik unik yang tidak ada dalam data lama"
            ],
            'suggestions': [
                "Tambahkan sinonim pada query",
                "Perluas dataset training dengan kasus serupa",
                "Coba metode embedding yang berbeda",
                "Gunakan teknik query expansion"
            ]
        })

    # Simpan ke file
    analysis_path = os.path.join(output_dir, f'retrieval_failure_analysis_{method_name}.json')
    with open(analysis_path, 'w', encoding='utf-8') as f:
        json.dump(analysis, f, indent=2, ensure_ascii=False)

    print(f"Analisis kegagalan retrieval disimpan di: {analysis_path}")
    return analysis

# 12. Analisis kegagalan prediksi
def analyze_prediction_failures(pred_metrics: pd.DataFrame, method_name: str, output_dir: str):
    # Identifikasi prediksi yang salah
    failed_predictions = pred_metrics[pred_metrics['correct'] == 0]

    # Ambil sampel kegagalan
    if not failed_predictions.empty:
        sample_failures = failed_predictions.sample(min(3, len(failed_predictions)))
    else:
        sample_failures = pd.DataFrame()

    # Simpan analisis dalam format JSON
    analysis = []
    for _, row in sample_failures.iterrows():
        analysis.append({
            'query_id': row['query_id'],
            'predicted_solution': row['predicted_solution'],
            'ground_truth_solution': row['ground_truth_solution'],
            'potential_causes': [
                "Kasus referensi tidak cukup relevan",
                "Metode agregasi solusi tidak optimal",
                "Variasi solusi terlalu tinggi pada kasus serupa",
                "Representasi teks tidak menangkap konteks hukum"
            ],
            'suggestions': [
                "Tingkatkan jumlah kasus yang di-retrieve (k)",
                "Coba metode agregasi berbeda (majority vs weighted)",
                "Gunakan teknik re-ranking berbasis solusi",
                "Terapkan fine-tuning model pada domain hukum"
            ]
        })

    # Simpan ke file
    analysis_path = os.path.join(output_dir, f'prediction_failure_analysis_{method_name}.json')
    with open(analysis_path, 'w', encoding='utf-8') as f:
        json.dump(analysis, f, indent=2, ensure_ascii=False)

    print(f"Analisis kegagalan prediksi disimpan di: {analysis_path}")
    return analysis

# 13. Main execution
if __name__ == "__main__":
    # Konfigurasi - PERBAIKAN PATH
    BASE_DIR = "/content/drive/MyDrive/TGS PK/data"
    PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
    EVAL_DIR = os.path.join(BASE_DIR, "eval")
    RESULTS_DIR = os.path.join(BASE_DIR, "results")
    DATA_PATH = os.path.join(PROCESSED_DIR, "cases.json")
    QUERIES_PATH = os.path.join(EVAL_DIR, "queries.json")
    TEST_SIZE = 0.2
    TOP_K = 5

    # Pastikan direktori ada
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    os.makedirs(EVAL_DIR, exist_ok=True)
    os.makedirs(RESULTS_DIR, exist_ok=True)

    # Muat data
    print(f"\n{'='*50}")
    print("Memuat data...")
    df = load_data(DATA_PATH)
    print(f"Data dimuat: {len(df)} baris")

    # Jika tidak ada data, buat dan simpan data dummy
    if df.empty:
        print("Data kosong, membuat data dummy...")
        df = create_dummy_data()
        df.to_json(DATA_PATH, orient='records', indent=2, force_ascii=False)
        print(f"Data dummy disimpan di {DATA_PATH}")
    elif not os.path.exists(DATA_PATH):
        # Simpan data jika belum ada
        df.to_json(DATA_PATH, orient='records', indent=2, force_ascii=False)
        print(f"Data disimpan di {DATA_PATH}")

    # Validasi case_id yang ada
    all_case_ids = set(df['case_id'])
    print(f"Jumlah kasus yang tersedia: {len(df)}")

    # Split data hanya untuk evaluasi solusi
    print("\nMembagi data untuk evaluasi...")
    train_df, test_df = split_data(df, test_size=TEST_SIZE)

    # Muat query evaluasi
    if os.path.exists(QUERIES_PATH):
        print(f"\nMemuat query evaluasi dari {QUERIES_PATH}")
        with open(QUERIES_PATH, 'r', encoding='utf-8') as f:
            sample_queries = json.load(f)
    else:
        # Buat queries yang sesuai dengan data dummy
        print("\nFile queries.json tidak ditemukan, membuat queries yang sesuai dengan data")

        # Daftar query template dengan kata kunci terkait
        query_templates = [
            ("Q1", "Kasus pencurian dengan kekerasan oleh kelompok bersenjata", ["pencurian", "kekerasan", "bersenjata"]),
            ("Q2", "Perkara perdata tentang wanprestasi dalam kontrak konstruksi", ["wanprestasi", "kontrak", "konstruksi"]),
            ("Q3", "Pidana narkotika jenis sabu dengan jumlah besar", ["narkotika", "sabu", "jumlah besar"]),
            ("Q4", "Sengketa tanah warisan antar saudara", ["tanah", "warisan", "saudara"]),
            ("Q5", "Penganiayaan berat yang mengakibatkan kematian", ["penganiayaan", "berat", "kematian"]),
            ("Q6", "Perbuatan melawan hukum oleh perusahaan terhadap konsumen", ["melawan hukum", "perusahaan", "konsumen"]),
            ("Q7", "Pencemaran nama baik melalui media sosial", ["pencemaran", "nama baik", "media sosial"]),
            ("Q8", "Pemalsuan dokumen untuk pengajuan kredit bank", ["pemalsuan", "dokumen", "kredit"]),
            ("Q9", "Sengketa hak asuh anak setelah perceraian", ["hak asuh", "anak", "perceraian"]),
            ("Q10", "Korupsi pengadaan barang di instansi pemerintah", ["korupsi", "pengadaan", "pemerintah"])
        ]

        sample_queries = []
        for qid, qtext, keywords in query_templates:
            relevant_cases = []

            # Cari kasus yang cocok dengan kata kunci
            for _, row in df.iterrows():
                text = row['text_full'].lower()
                if any(keyword in text for keyword in keywords):
                    relevant_cases.append(row['case_id'])
                    if len(relevant_cases) >= 2:
                        break

            # Jika tidak ada yang cocok, ambil secara acak
            if len(relevant_cases) < 2:
                additional = random.sample(df['case_id'].tolist(), 2 - len(relevant_cases))
                relevant_cases.extend(additional)

            sample_queries.append({
                "query_id": qid,
                "query_text": qtext,
                "relevant_cases": relevant_cases[:2]  # Ambil maksimal 2 kasus
            })

        # Simpan queries
        with open(QUERIES_PATH, 'w', encoding='utf-8') as f:
            json.dump(sample_queries, f, indent=2, ensure_ascii=False)
        print(f"Sample queries disimpan di {QUERIES_PATH}")

    # Validasi semua relevant_cases ada dalam data
    all_case_ids = set(df['case_id'])
    for query in sample_queries:
        valid_cases = [case_id for case_id in query['relevant_cases'] if case_id in all_case_ids]
        if len(valid_cases) < len(query['relevant_cases']):
            print(f"Peringatan: Beberapa case ID tidak ditemukan dalam data untuk query {query['query_id']}")
            # Tambahkan kasus valid jika kurang
            if len(valid_cases) < 2:
                additional = random.sample(list(all_case_ids - set(valid_cases)), 2 - len(valid_cases))
                valid_cases.extend(additional)
        query['relevant_cases'] = valid_cases[:2]  # Pastikan hanya 2 kasus relevan

    # Evaluasi untuk berbagai metode
    all_retrieval_metrics = []
    all_prediction_metrics = []
    model_summary = []
    retrieval_methods = ['tfidf', 'bert']  # Hanya gunakan metode yang terbukti efektif
    prediction_methods = ['majority', 'weighted']

    for method in retrieval_methods:
        print(f"\n{'='*50}")
        print(f"EVALUASI METODE: {method.upper()}")
        print(f"{'='*50}")

        # Inisialisasi vectorizer dengan SELURUH data
        vectorizer = Vectorizer(method=method)
        print("Fitting vectorizer...")
        vectorizer.fit(df['text_full'])  # Gunakan seluruh data

        print("Transforming data...")
        all_vectors = vectorizer.transform(df['text_full'])
        print(f"Vektor dibuat: {all_vectors.shape}")

        # Inisialisasi retriever dengan SELURUH data
        retriever = CaseRetriever(
            vectorizer=vectorizer,
            case_ids=df['case_id'].tolist(),
            vectors=all_vectors
        )

        # Buat mapping solusi untuk seluruh data
        case_solutions = dict(zip(df['case_id'], df['solution']))

        # Evaluasi retrieval
        print("Evaluasi retrieval...")
        retrieval_metrics = eval_retrieval(sample_queries, retriever, k=TOP_K)
        retrieval_metrics['method'] = method

        # Simpan metrik retrieval
        retrieval_metrics_path = os.path.join(EVAL_DIR, f"retrieval_metrics_{method}.csv")
        retrieval_metrics.to_csv(retrieval_metrics_path, index=False)
        print(f"Metrik retrieval disimpan di {retrieval_metrics_path}")

        # Visualisasi metrik per query
        visualize_retrieval_metrics(retrieval_metrics, method, EVAL_DIR)

        # Analisis kegagalan retrieval
        analyze_retrieval_failures(retrieval_metrics, method, EVAL_DIR)

        # Simpan ringkasan metrik untuk model ini
        avg_metrics = retrieval_metrics[['precision', 'recall', 'f1', 'accuracy']].mean().to_dict()
        model_summary.append({
            'method': method,
            'precision': avg_metrics['precision'],
            'recall': avg_metrics['recall'],
            'f1': avg_metrics['f1'],
            'accuracy': avg_metrics['accuracy']
        })

        # Evaluasi prediksi untuk semua metode
        for pred_method in prediction_methods:
            print(f"Evaluasi prediksi dengan metode {pred_method}...")
            prediction_metrics = eval_prediction(
                sample_queries, retriever, case_solutions, method=pred_method
            )
            prediction_metrics['retrieval_method'] = method
            prediction_metrics['prediction_method'] = pred_method

            # Hitung akurasi
            accuracy = prediction_metrics['correct'].mean()
            print(f"Akurasi prediksi ({method}/{pred_method}): {accuracy:.2f}")

            # Analisis kegagalan prediksi
            analyze_prediction_failures(prediction_metrics, f"{method}_{pred_method}", EVAL_DIR)

            all_prediction_metrics.append(prediction_metrics)

        all_retrieval_metrics.append(retrieval_metrics)

    # Gabungkan dan simpan semua metrik retrieval
    final_retrieval_metrics = pd.concat(all_retrieval_metrics)
    final_retrieval_metrics_path = os.path.join(EVAL_DIR, "retrieval_metrics.csv")
    final_retrieval_metrics.to_csv(final_retrieval_metrics_path, index=False)

    # Gabungkan dan simpan semua metrik prediksi
    final_prediction_metrics = pd.concat(all_prediction_metrics)
    final_prediction_metrics_path = os.path.join(EVAL_DIR, "prediction_metrics.csv")
    final_prediction_metrics.to_csv(final_prediction_metrics_path, index=False)

    # Buat DataFrame ringkasan model
    model_comparison_df = pd.DataFrame(model_summary)

    # Visualisasi perbandingan model retrieval
    visualize_model_comparison(model_comparison_df, EVAL_DIR)

    # Ringkasan metrik prediksi
    pred_summary = final_prediction_metrics.groupby(
        ['retrieval_method', 'prediction_method']
    )['correct'].agg(['mean', 'count']).reset_index()
    pred_summary.rename(columns={'mean': 'accuracy'}, inplace=True)

    # Visualisasi perbandingan metode prediksi
    visualize_prediction_comparison(pred_summary, EVAL_DIR)

    # Simpan ringkasan model
    model_comparison_path = os.path.join(EVAL_DIR, "model_comparison.csv")
    model_comparison_df.to_csv(model_comparison_path, index=False)

    print("\n=== HASIL EVALUASI ===")
    print(f"Metrik evaluasi retrieval disimpan di: {final_retrieval_metrics_path}")
    print(f"Metrik evaluasi prediksi disimpan di: {final_prediction_metrics_path}")
    print(f"Ringkasan perbandingan model disimpan di: {model_comparison_path}")

    # Laporan ringkasan
    print("\nRINGKASAN METRIK RETRIEVAL:")
    print(model_comparison_df)

    print("\nRINGKASAN METRIK PREDIKSI (Akurasi):")
    print(pred_summary)

    # Simpan ringkasan teks
    summary_path = os.path.join(EVAL_DIR, "evaluation_summary.txt")
    with open(summary_path, 'w', encoding='utf-8') as f:
        f.write("=== EVALUASI SISTEM RETRIEVAL PUTUSAN PENGADILAN ===\n\n")
        f.write("RINGKASAN METRIK RETRIEVAL:\n")
        f.write(model_comparison_df.to_string() + "\n\n")
        f.write("RINGKASAN METRIK PREDIKSI (Akurasi):\n")
        f.write(pred_summary.to_string() + "\n\n")
        f.write("ANALISIS KEGAGALAN:\n")
        f.write("- File analisis kegagalan tersedia di direktori eval dengan format:\n")
        f.write("  - retrieval_failure_analysis_<method>.json\n")
        f.write("  - prediction_failure_analysis_<method>_<pred_method>.json\n")
        f.write("- Visualisasi metrik tersedia di direktori eval\n")

    print(f"\nRingkasan evaluasi lengkap disimpan di: {summary_path}")
    print("\nEvaluasi selesai!")


Memuat data...
Kolom 'solution' tidak ditemukan. Membuat solusi dummy...
Menggunakan 'argumen_hukum' sebagai solution
Data dimuat: 30 baris
Jumlah kasus yang tersedia: 30

Membagi data untuk evaluasi...

Memuat query evaluasi dari /content/drive/MyDrive/TGS PK/data/eval/queries.json
Peringatan: Beberapa case ID tidak ditemukan dalam data untuk query Q1
Peringatan: Beberapa case ID tidak ditemukan dalam data untuk query Q2
Peringatan: Beberapa case ID tidak ditemukan dalam data untuk query Q3
Peringatan: Beberapa case ID tidak ditemukan dalam data untuk query Q4
Peringatan: Beberapa case ID tidak ditemukan dalam data untuk query Q5
Peringatan: Beberapa case ID tidak ditemukan dalam data untuk query Q6
Peringatan: Beberapa case ID tidak ditemukan dalam data untuk query Q7
Peringatan: Beberapa case ID tidak ditemukan dalam data untuk query Q8
Peringatan: Beberapa case ID tidak ditemukan dalam data untuk query Q9
Peringatan: Beberapa case ID tidak ditemukan dalam data untuk query Q10

EVA

<Figure size 1400x800 with 0 Axes>

<Figure size 1400x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>